# **Task 0**

In [1]:
import requests
import random


#### PROBLEM 2, TASK 0: DATASET PREPARATION (STRICTLY UNIQUE ENSURED FOR IT, FIRST NAME) ####


print("Fetching 1000 STRICTLY UNIQUE single-word Indian names...")

# Using the mbejda GitHub Gist datasets (~28,000 real Indian names)
urls = [
    "https://gist.githubusercontent.com/mbejda/9b93c7545c9dd93060bd/raw/Indian-Female-Names.csv",
    "https://gist.githubusercontent.com/mbejda/7f86ca901fe41bc14a63/raw/Indian-Male-Names.csv"
]

unique_single_names = set()

try:
    for url in urls:
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        # Parse the CSV text line by line
        lines = response.text.split('\n')
        for line in lines[1:]:  # Skip the header row
            parts = line.split(',')
            if parts:
                raw_full_name = parts[0].strip().title()

                # Extract ONLY the first name
                first_name = raw_full_name.split(' ')[0]

                # Filter out garbage data (must be purely letters, >2 chars)
                if first_name.isalpha() and len(first_name) > 2:
                    unique_single_names.add(first_name)

    # We now have thousands of completely unique words. Sample exactly 1000.
    sampled_names = random.sample(list(unique_single_names), 1000)

    # Sort alphabetically for a clean text file
    final_names_list = sorted(sampled_names)

    # Save strictly as "Training Names.txt"
    file_name = "Training Names.txt"
    with open(file_name, "w", encoding="utf-8") as f:
        for name in final_names_list:
            f.write(name + "\n")

    print(f"Success! Saved {len(final_names_list)} perfectly unique names.")
    print(f"Saved to '{file_name}'.")

    print("\nSample of generated names (Notice: Zero repetition):")
    for i in range(15):
        print(f" - {final_names_list[i]}")

except Exception as e:
    print(f"Error fetching data: {e}")

Fetching 1000 STRICTLY UNIQUE single-word Indian names...
Success! Saved 1000 perfectly unique names.
Saved to 'Training Names.txt'.

Sample of generated names (Notice: Zero repetition):
 - Aabid
 - Aachal
 - Aafrin
 - Aakash
 - Aarif
 - Aarti
 - Aasha
 - Aashik
 - Aashis
 - Aashiya
 - Aashu
 - Aasu
 - Abash
 - Abdulla
 - Abhash


# **Task 1**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F


#### TASK 1: HYPERPARAMETERS & SETUP ####

# Specifying hyperparameters as strictly required by the assignment
EMBEDDING_DIM = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 1
BATCH_SIZE = 128
LEARNING_RATE = 0.005
EPOCHS = 500

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


#### DATA PREPARATION (Character Level) ####

with open("Training Names.txt", "r", encoding="utf-8") as f:
    names = [line.strip().lower() for line in f.readlines()]

# Create character vocabulary
chars = set(''.join(names))
# Add special tokens: Padding, Start of Sequence, End of Sequence
PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2
vocab = ['<PAD>', '<SOS>', '<EOS>'] + sorted(list(chars))
char2idx = {ch: i for i, ch in enumerate(vocab)}
idx2char = {i: ch for i, ch in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

# Find the longest name to pad all sequences to the same length
max_len = max(len(name) for name in names) + 2 # +2 for <SOS> and <EOS>

class NameDataset(Dataset):
    """Formats names into tokenized, padded sequences for the DataLoader."""
    def __init__(self, names, char2idx, max_len):
        self.inputs = []
        self.targets = []

        for name in names:
            # Convert characters to indices and add SOS/EOS tokens
            indices = [char2idx['<SOS>']] + [char2idx[ch] for ch in name] + [char2idx['<EOS>']]

            # Input is everything except the last character, Target is everything except the first
            inp = indices[:-1]
            tgt = indices[1:]

            # Pad sequences to max length for batching
            inp += [char2idx['<PAD>']] * (max_len - len(inp) - 1)
            tgt += [char2idx['<PAD>']] * (max_len - len(tgt) - 1)

            self.inputs.append(inp)
            self.targets.append(tgt)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx], dtype=torch.long), torch.tensor(self.targets[idx], dtype=torch.long)

dataset = NameDataset(names, char2idx, max_len)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)


# MODEL 1: VANILLA RNN

class VanillaRNN(nn.Module):
    """Architecture: Embedding Layer -> Standard RNN Layer -> Fully Connected Linear Layer."""
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers):
        super(VanillaRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.RNN(embed_dim, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        out = self.embedding(x)
        out, hidden = self.rnn(out, hidden)
        out = self.fc(out)
        return out, hidden


# MODEL 2: BIDIRECTIONAL LSTM (BLSTM)

class BLSTM(nn.Module):
    """Architecture: Embedding Layer -> Bidirectional LSTM Layer -> Fully Connected Linear Layer.
       The linear layer takes 2 * hidden_size because it concatenates forward and backward states."""
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers):
        super(BLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        # Set bidirectional=True
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True, bidirectional=True)
        # Multiply hidden_size by 2 for the output fully connected layer
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, hidden=None):
        out = self.embedding(x)
        out, hidden = self.lstm(out, hidden)
        out = self.fc(out)
        return out, hidden


# MODEL 3: RNN WITH BASIC ATTENTION


class AttentionRNN(nn.Module):
    """Architecture: Embedding Layer -> Standard RNN -> Dot-Product Attention -> Linear Layer."""
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers):
        super(AttentionRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.RNN(embed_dim, hidden_size, num_layers, batch_first=True)

        # Attention mechanisms
        self.attention_weights = nn.Linear(hidden_size, 1, bias=False)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        rnn_out, hidden = self.rnn(embedded, hidden) # rnn_out: [batch, seq_len, hidden_size]

        # Basic Attention Mechanism (Self-Attention over the sequence)
        # Calculate alignment scores
        attn_scores = self.attention_weights(rnn_out) # [batch, seq_len, 1]
        attn_weights = F.softmax(attn_scores, dim=1)  # [batch, seq_len, 1]

        # Apply attention weights to RNN outputs
        context_vector = rnn_out * attn_weights # Element-wise multiplication

        # Pass the context-aware output to the final linear layer
        out = self.fc(context_vector)
        return out, hidden


# INSTANTIATION & PARAMETER COUNTING

def count_parameters(model):
    """Helper function to report trainable parameters as required."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

models = {
    "Vanilla RNN": VanillaRNN(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS).to(device),
    "BLSTM": BLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS).to(device),
    "Attention RNN": AttentionRNN(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS).to(device)
}

print("\n--- Model Architectures & Trainable Parameters ---")
for name, model in models.items():
    print(f"{name}: {count_parameters(model):,} trainable parameters")


# TRAINING LOOP

print("\n--- Starting Training ---")
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX) # Ignore padding in loss calculation

for name, model in models.items():
    print(f"\nTraining {name}...")
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs, _ = model(inputs)

            # Reshape for CrossEntropyLoss: [batch_size * seq_len, vocab_size]
            loss = criterion(outputs.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(dataloader):.4f}")

Using device: cuda

--- Model Architectures & Trainable Parameters ---
Vanilla RNN: 30,429 trainable parameters
BLSTM: 207,965 trainable parameters
Attention RNN: 30,557 trainable parameters

--- Starting Training ---

Training Vanilla RNN...
  Epoch 1/500 | Loss: 2.8187
  Epoch 10/500 | Loss: 1.9874
  Epoch 20/500 | Loss: 1.7628
  Epoch 30/500 | Loss: 1.5558
  Epoch 40/500 | Loss: 1.3869
  Epoch 50/500 | Loss: 1.2620
  Epoch 60/500 | Loss: 1.1673
  Epoch 70/500 | Loss: 1.1208
  Epoch 80/500 | Loss: 1.0920
  Epoch 90/500 | Loss: 1.0610
  Epoch 100/500 | Loss: 1.0423
  Epoch 110/500 | Loss: 1.0333
  Epoch 120/500 | Loss: 1.0205
  Epoch 130/500 | Loss: 1.0174
  Epoch 140/500 | Loss: 1.0155
  Epoch 150/500 | Loss: 1.0035
  Epoch 160/500 | Loss: 0.9975
  Epoch 170/500 | Loss: 0.9950
  Epoch 180/500 | Loss: 0.9915
  Epoch 190/500 | Loss: 0.9908
  Epoch 200/500 | Loss: 0.9903
  Epoch 210/500 | Loss: 0.9908
  Epoch 220/500 | Loss: 0.9825
  Epoch 230/500 | Loss: 0.9841
  Epoch 240/500 | Loss: 

# **Task 2**

In [3]:
import torch.nn.functional as F


# TASK 2: QUANTITATIVE EVALUATION


# 1. Load the original training names into a Set for fast Novelty checking
training_names_set = set(names)  # 'names' is the list from Task 1
NUM_NAMES_TO_GENERATE = 500      # Generate 500 names per model for a solid sample size
GENERATION_TEMP = 0.8            # Temperature for sampling (0.8 adds slight randomness to prevent loops)

def generate_name(model, start_char='<SOS>', max_length=20):
    """Generates a single name character-by-character autoregressively."""
    model.eval() # Set to evaluation mode
    with torch.no_grad():
        # Start the sequence with the <SOS> token
        current_seq = [char2idx[start_char]]

        for _ in range(max_length):
            # Convert current sequence to tensor and add batch dimension
            inputs = torch.tensor([current_seq], dtype=torch.long).to(device)

            # Forward pass
            outputs, _ = model(inputs)

            # Get the logits for the LAST character in the sequence
            next_char_logits = outputs[0, -1, :]

            # Apply temperature scaling to logits
            next_char_logits = next_char_logits / GENERATION_TEMP

            # Convert logits to probabilities
            probs = F.softmax(next_char_logits, dim=0)

            # Sample the next character from the probability distribution
            next_char_idx = torch.multinomial(probs, 1).item()

            # Stop if the model generates the End Of Sequence token
            if next_char_idx == char2idx['<EOS>']:
                break

            # Avoid appending padding tokens
            if next_char_idx != char2idx['<PAD>']:
                current_seq.append(next_char_idx)

        # Convert indices back to string (skipping the initial <SOS> token)
        generated_string = ''.join([idx2char[idx] for idx in current_seq[1:]])
        return generated_string.capitalize()

def evaluate_model(model, model_name):
    """Generates names and calculates Novelty and Diversity rates."""
    print(f"\nEvaluating {model_name}...")

    generated_names = []
    for _ in range(NUM_NAMES_TO_GENERATE):
        name = generate_name(model)
        # Filter out empty names if the model collapsed
        if len(name) > 1:
            generated_names.append(name)

    # Calculate Metrics
    total_generated = len(generated_names)
    if total_generated == 0:
        return 0.0, 0.0

    unique_generated = set(generated_names)

    # Novelty: Names NOT in the training set
    novel_names = [name for name in generated_names if name.lower() not in training_names_set]

    diversity_rate = (len(unique_generated) / total_generated) * 100
    novelty_rate = (len(novel_names) / total_generated) * 100

    print(f"  Total Valid Names Generated: {total_generated}/{NUM_NAMES_TO_GENERATE}")
    print(f"  Unique Names Generated:      {len(unique_generated)}")
    print(f"  Novel Names (Not in Train):  {len(novel_names)}")
    print(f"  --> Diversity Rate:          {diversity_rate:.2f}%")
    print(f"  --> Novelty Rate:            {novelty_rate:.2f}%")

    return generated_names

# 2. Run the evaluation for all three models
generated_results = {}

print("--- Quantitative Evaluation Results ---")
for model_name, model in models.items():
    generated_results[model_name] = evaluate_model(model, model_name)

print("\nEvaluation complete. Use these percentages in your report.")

--- Quantitative Evaluation Results ---

Evaluating Vanilla RNN...
  Total Valid Names Generated: 500/500
  Unique Names Generated:      370
  Novel Names (Not in Train):  7
  --> Diversity Rate:          74.00%
  --> Novelty Rate:            1.40%

Evaluating BLSTM...
  Total Valid Names Generated: 498/500
  Unique Names Generated:      494
  Novel Names (Not in Train):  495
  --> Diversity Rate:          99.20%
  --> Novelty Rate:            99.40%

Evaluating Attention RNN...
  Total Valid Names Generated: 500/500
  Unique Names Generated:      49
  Novel Names (Not in Train):  80
  --> Diversity Rate:          9.80%
  --> Novelty Rate:            16.00%

Evaluation complete. Use these percentages in your report.


# **Task 3**

In [4]:
import random


# TASK 3: QUALITATIVE ANALYSIS SAMPLES


print("--- Representative Generated Samples ---")

# We use the generated_results dictionary from Task 2
for model_name, names_list in generated_results.items():
    print(f"\nModel: {model_name}")

    # Ensure we have enough names to sample
    if len(names_list) >= 10:
        samples = random.sample(names_list, 10)
    else:
        samples = names_list

    for i, name in enumerate(samples):
        print(f"  {i+1}. {name}")

--- Representative Generated Samples ---

Model: Vanilla RNN
  1. Saira
  2. Afshana
  3. Fool
  4. Amrjeet
  5. Nanshi
  6. Nanku
  7. Muskarn
  8. Hemant
  9. Irfa
  10. Naseeba

Model: BLSTM
  1. Ulsha
  2. Puljool
  3. Yooggan
  4. Gohoyan
  5. Luxfoov
  6. Unajm
  7. Bolaj
  8. Bulu
  9. Yandeesaa
  10. Uveer

Model: Attention RNN
  1. Aasha
  2. Aasha
  3. Aasha
  4. Aashi
  5. Aana
  6. Aasha
  7. Aasha
  8. Aasha
  9. Aashana
  10. Aasha
